# Setup: Synthetic Patient Data for HIPAA Lab

**Purpose:** This notebook generates a realistic synthetic healthcare dataset and loads it into Unity Catalog.

## What Gets Created
| Table | Records | Description |
|-------|---------|-------------|
| `patients` | 10,000 | Patient demographics with PHI (name, SSN, DOB, address, phone, email) |
| `encounters` | ~50,000 | Hospital visit records (ER, inpatient, outpatient, telehealth) |
| `diagnoses` | ~80,000 | ICD-10 diagnosis codes linked to encounters |
| `lab_results` | ~40,000 | Laboratory test results (blood panels, metabolic, etc.) |
| `medications` | ~30,000 | Prescription records linked to patients and encounters |

## Target Catalog & Schema
- **Catalog:** `hipaa_demo`
- **Schema:** `hipaa_demo.clinical_data`

> ⏱ **Estimated runtime:** ~3-5 minutes on serverless compute

---
**Run all cells in order to set up the lab environment.**

In [0]:
# =============================================================================
# Step 1: Create the Unity Catalog and Schema for our healthcare demo
# =============================================================================

spark.sql("CREATE CATALOG IF NOT EXISTS hipaa_demo")
spark.sql("USE CATALOG hipaa_demo")

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS clinical_data
    COMMENT 'HIPAA-compliant clinical data schema for patient analytics demo'
""")

spark.sql("USE SCHEMA clinical_data")

print("✅ Catalog 'hipaa_demo' and schema 'clinical_data' created successfully.")
print(f"   Current catalog: {spark.sql('SELECT current_catalog()').collect()[0][0]}")
print(f"   Current schema:  {spark.sql('SELECT current_schema()').collect()[0][0]}")

In [0]:
# =============================================================================
# Step 2: Generate 10,000 synthetic patient records with realistic PHI
# =============================================================================
import random
import string
from datetime import datetime, timedelta
from pyspark.sql.types import *
from pyspark.sql import functions as F

random.seed(42)
NUM_PATIENTS = 10000

# --- Reference data ---
first_names_m = ["James","John","Robert","Michael","David","William","Richard","Joseph","Thomas","Charles",
                 "Christopher","Daniel","Matthew","Anthony","Mark","Donald","Steven","Andrew","Paul","Joshua",
                 "Kenneth","Kevin","Brian","George","Timothy","Ronald","Edward","Jason","Jeffrey","Ryan"]
first_names_f = ["Mary","Patricia","Jennifer","Linda","Barbara","Elizabeth","Susan","Jessica","Sarah","Karen",
                 "Lisa","Nancy","Betty","Margaret","Sandra","Ashley","Dorothy","Kimberly","Emily","Donna",
                 "Michelle","Carol","Amanda","Melissa","Deborah","Stephanie","Rebecca","Sharon","Laura","Cynthia"]
last_names = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez",
              "Hernandez","Lopez","Gonzalez","Wilson","Anderson","Thomas","Taylor","Moore","Jackson","Martin",
              "Lee","Perez","Thompson","White","Harris","Sanchez","Clark","Ramirez","Lewis","Robinson",
              "Walker","Young","Allen","King","Wright","Scott","Torres","Nguyen","Hill","Flores"]

street_names = ["Main St","Oak Ave","Elm St","Park Blvd","Cedar Ln","Maple Dr","Pine Rd","Washington St",
                "Lake Ave","Hill Rd","River Dr","Sunset Blvd","Spring St","Valley Rd","Forest Ave"]
cities_states = [
    ("New York","NY","10001"),("Los Angeles","CA","90001"),("Chicago","IL","60601"),("Houston","TX","77001"),
    ("Phoenix","AZ","85001"),("Philadelphia","PA","19101"),("San Antonio","TX","78201"),("San Diego","CA","92101"),
    ("Dallas","TX","75201"),("Austin","TX","73301"),("Jacksonville","FL","32099"),("San Francisco","CA","94101"),
    ("Columbus","OH","43085"),("Indianapolis","IN","46201"),("Charlotte","NC","28201"),("Seattle","WA","98101"),
    ("Denver","CO","80201"),("Boston","MA","02101"),("Nashville","TN","37201"),("Portland","OR","97201")
]

races = ["White","Black or African American","Asian","Hispanic or Latino","Native American","Pacific Islander","Other"]
race_weights = [0.40, 0.18, 0.12, 0.20, 0.03, 0.02, 0.05]
ethnicities = ["Not Hispanic or Latino","Hispanic or Latino"]
genders = ["Male","Female","Non-binary"]
gender_weights = [0.49, 0.49, 0.02]
insurance_types = ["Medicare","Medicaid","Blue Cross","Aetna","UnitedHealth","Cigna","Humana","Self-Pay"]
insurance_weights = [0.25, 0.20, 0.15, 0.10, 0.12, 0.08, 0.05, 0.05]
blood_types = ["A+","A-","B+","B-","AB+","AB-","O+","O-"]
blood_weights = [0.34, 0.06, 0.09, 0.02, 0.03, 0.01, 0.38, 0.07]

# Healthcare regions / departments (for row-level security demo later)
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West"]
departments = ["Cardiology", "Oncology", "Neurology", "Orthopedics", "Emergency", "Pediatrics", "Internal Medicine", "Surgery"]

def generate_ssn():
    return f"{random.randint(100,999)}-{random.randint(10,99)}-{random.randint(1000,9999)}"

def generate_phone():
    return f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}"

def generate_dob():
    start = datetime(1940, 1, 1)
    end = datetime(2006, 12, 31)
    return start + timedelta(days=random.randint(0, (end - start).days))

def generate_email(first, last, idx):
    domains = ["gmail.com","yahoo.com","outlook.com","hotmail.com","aol.com","icloud.com"]
    return f"{first.lower()}.{last.lower()}{idx}@{random.choice(domains)}"

patients = []
for i in range(1, NUM_PATIENTS + 1):
    gender = random.choices(genders, weights=gender_weights)[0]
    if gender == "Male":
        first = random.choice(first_names_m)
    elif gender == "Female":
        first = random.choice(first_names_f)
    else:
        first = random.choice(first_names_m + first_names_f)
    last = random.choice(last_names)
    city, state, zipcode = random.choice(cities_states)
    dob = generate_dob()

    patients.append((
        f"PAT-{i:06d}",                                    # patient_id
        first,                                               # first_name
        last,                                                # last_name
        generate_ssn(),                                      # ssn
        dob.strftime("%Y-%m-%d"),                            # date_of_birth
        gender,                                              # gender
        random.choices(races, weights=race_weights)[0],      # race
        random.choices(ethnicities, weights=[0.80, 0.20])[0],# ethnicity
        f"{random.randint(100,9999)} {random.choice(street_names)}",  # address
        city,                                                # city
        state,                                               # state
        zipcode,                                             # zip_code
        generate_phone(),                                    # phone_number
        generate_email(first, last, i),                      # email
        random.choices(insurance_types, weights=insurance_weights)[0],  # insurance_type
        f"INS-{random.randint(100000000,999999999)}",        # insurance_id
        random.choices(blood_types, weights=blood_weights)[0],# blood_type
        random.choice(regions),                              # region
        random.choice(departments),                          # primary_department
        (datetime(2020,1,1) + timedelta(days=random.randint(0,1800))).strftime("%Y-%m-%d"),  # registration_date
        random.choice([True, False]) if random.random() < 0.05 else True  # is_active
    ))

patient_schema = StructType([
    StructField("patient_id", StringType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("ssn", StringType()),
    StructField("date_of_birth", StringType()),
    StructField("gender", StringType()),
    StructField("race", StringType()),
    StructField("ethnicity", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("zip_code", StringType()),
    StructField("phone_number", StringType()),
    StructField("email", StringType()),
    StructField("insurance_type", StringType()),
    StructField("insurance_id", StringType()),
    StructField("blood_type", StringType()),
    StructField("region", StringType()),
    StructField("primary_department", StringType()),
    StructField("registration_date", StringType()),
    StructField("is_active", BooleanType())
])

df_patients = spark.createDataFrame(patients, schema=patient_schema) \
    .withColumn("date_of_birth", F.col("date_of_birth").cast("date")) \
    .withColumn("registration_date", F.col("registration_date").cast("date"))

print(f"✅ Generated {df_patients.count():,} patient records")
df_patients.show(5, truncate=False)

In [0]:
# =============================================================================
# Step 3: Generate ~50,000 encounter records linked to patients
# =============================================================================
import uuid

random.seed(123)

encounter_types = ["Emergency", "Inpatient", "Outpatient", "Telehealth", "Urgent Care"]
encounter_weights = [0.15, 0.20, 0.40, 0.15, 0.10]
discharge_dispositions = ["Home", "Home Health", "SNF", "Rehab", "AMA", "Expired", "Transfer"]
discharge_weights = [0.50, 0.15, 0.10, 0.08, 0.05, 0.02, 0.10]
facilities = ["Main Hospital", "North Campus", "South Campus", "Downtown Clinic", "East Wing", "West Pavilion"]

patient_ids = [f"PAT-{i:06d}" for i in range(1, NUM_PATIENTS + 1)]

encounters = []
enc_counter = 0
for pid in patient_ids:
    # Each patient gets 1-15 encounters
    num_enc = random.choices(range(1, 16), weights=[10,15,15,12,10,8,7,6,5,4,3,2,1,1,1])[0]
    for _ in range(num_enc):
        enc_counter += 1
        enc_type = random.choices(encounter_types, weights=encounter_weights)[0]
        admit_date = datetime(2021, 1, 1) + timedelta(days=random.randint(0, 1800))

        # Length of stay depends on encounter type
        if enc_type == "Inpatient":
            los = random.randint(1, 21)
        elif enc_type == "Emergency":
            los = random.randint(0, 3)
        elif enc_type == "Urgent Care":
            los = 0
        else:
            los = 0

        discharge_date = admit_date + timedelta(days=los)
        dept = random.choice(departments)
        region = random.choice(regions)

        encounters.append((
            f"ENC-{enc_counter:08d}",                          # encounter_id
            pid,                                                # patient_id
            enc_type,                                           # encounter_type
            admit_date.strftime("%Y-%m-%d"),                    # admit_date
            discharge_date.strftime("%Y-%m-%d"),                # discharge_date
            los,                                                # length_of_stay
            random.choices(discharge_dispositions, weights=discharge_weights)[0],  # discharge_disposition
            random.choice(facilities),                          # facility
            dept,                                               # department
            f"DR-{random.randint(1000,9999)}",                  # attending_physician_id
            region,                                             # region
            round(random.uniform(100, 150000), 2) if enc_type in ["Inpatient","Emergency"] else round(random.uniform(50, 5000), 2),  # total_charges
            random.choice(["Completed", "Completed", "Completed", "Cancelled", "No-Show"])  # status
        ))

encounter_schema = StructType([
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("encounter_type", StringType()),
    StructField("admit_date", StringType()),
    StructField("discharge_date", StringType()),
    StructField("length_of_stay", IntegerType()),
    StructField("discharge_disposition", StringType()),
    StructField("facility", StringType()),
    StructField("department", StringType()),
    StructField("attending_physician_id", StringType()),
    StructField("region", StringType()),
    StructField("total_charges", DoubleType()),
    StructField("status", StringType())
])

df_encounters = spark.createDataFrame(encounters, schema=encounter_schema) \
    .withColumn("admit_date", F.col("admit_date").cast("date")) \
    .withColumn("discharge_date", F.col("discharge_date").cast("date"))

print(f"✅ Generated {df_encounters.count():,} encounter records")
df_encounters.show(5, truncate=False)

In [0]:
# =============================================================================
# Step 4: Generate ~80,000 diagnosis records with ICD-10 codes
# =============================================================================
random.seed(456)

# Realistic ICD-10 codes and descriptions
icd10_codes = [
    ("I10", "Essential hypertension"),
    ("E11.9", "Type 2 diabetes mellitus without complications"),
    ("J06.9", "Acute upper respiratory infection"),
    ("M54.5", "Low back pain"),
    ("J18.9", "Pneumonia, unspecified organism"),
    ("I25.10", "Atherosclerotic heart disease"),
    ("E78.5", "Hyperlipidemia, unspecified"),
    ("F32.9", "Major depressive disorder, single episode"),
    ("K21.0", "Gastro-esophageal reflux disease with esophagitis"),
    ("N39.0", "Urinary tract infection, site not specified"),
    ("J45.909", "Unspecified asthma, uncomplicated"),
    ("G43.909", "Migraine, unspecified"),
    ("M79.3", "Panniculitis, unspecified"),
    ("R10.9", "Unspecified abdominal pain"),
    ("Z87.891", "Personal history of nicotine dependence"),
    ("E03.9", "Hypothyroidism, unspecified"),
    ("I48.91", "Unspecified atrial fibrillation"),
    ("J44.1", "COPD with acute exacerbation"),
    ("N18.3", "Chronic kidney disease, stage 3"),
    ("C50.919", "Malignant neoplasm of breast"),
    ("G47.33", "Obstructive sleep apnea"),
    ("F41.1", "Generalized anxiety disorder"),
    ("K80.20", "Calculus of gallbladder without cholecystitis"),
    ("M17.11", "Primary osteoarthritis, right knee"),
    ("D64.9", "Anemia, unspecified"),
    ("R51.9", "Headache, unspecified"),
    ("Z23", "Encounter for immunization"),
    ("B34.9", "Viral infection, unspecified"),
    ("L30.9", "Dermatitis, unspecified"),
    ("R05.9", "Cough, unspecified")
]

diagnosis_types = ["Principal", "Secondary", "Admitting", "Comorbidity"]
diag_type_weights = [0.30, 0.40, 0.10, 0.20]

enc_ids = [e[0] for e in encounters]
enc_patient_map = {e[0]: e[1] for e in encounters}

diagnoses = []
diag_counter = 0
for enc_id in enc_ids:
    # 1-4 diagnoses per encounter
    num_diag = random.choices([1, 2, 3, 4], weights=[0.25, 0.35, 0.25, 0.15])[0]
    used_codes = set()
    for j in range(num_diag):
        diag_counter += 1
        code, desc = random.choice(icd10_codes)
        while code in used_codes:
            code, desc = random.choice(icd10_codes)
        used_codes.add(code)

        diagnoses.append((
            f"DX-{diag_counter:08d}",                          # diagnosis_id
            enc_id,                                             # encounter_id
            enc_patient_map[enc_id],                            # patient_id
            code,                                               # icd10_code
            desc,                                               # diagnosis_description
            random.choices(diagnosis_types, weights=diag_type_weights)[0],  # diagnosis_type
            j + 1,                                              # diagnosis_rank
            random.choice(["Active", "Active", "Active", "Resolved", "Chronic"]),  # status
            (datetime(2021,1,1) + timedelta(days=random.randint(0,1800))).strftime("%Y-%m-%d")  # diagnosis_date
        ))

diagnosis_schema = StructType([
    StructField("diagnosis_id", StringType()),
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("icd10_code", StringType()),
    StructField("diagnosis_description", StringType()),
    StructField("diagnosis_type", StringType()),
    StructField("diagnosis_rank", IntegerType()),
    StructField("status", StringType()),
    StructField("diagnosis_date", StringType())
])

df_diagnoses = spark.createDataFrame(diagnoses, schema=diagnosis_schema) \
    .withColumn("diagnosis_date", F.col("diagnosis_date").cast("date"))

print(f"✅ Generated {df_diagnoses.count():,} diagnosis records")
df_diagnoses.show(5, truncate=False)

In [0]:
# =============================================================================
# Step 5: Generate ~40,000 lab result records
# =============================================================================
random.seed(789)

# Lab tests with realistic ranges
lab_tests = [
    ("CBC-WBC", "White Blood Cell Count", "10^3/uL", 4.5, 11.0, 0.5, 30.0),
    ("CBC-RBC", "Red Blood Cell Count", "10^6/uL", 4.0, 5.5, 2.0, 8.0),
    ("CBC-HGB", "Hemoglobin", "g/dL", 12.0, 17.5, 5.0, 22.0),
    ("CBC-PLT", "Platelet Count", "10^3/uL", 150.0, 400.0, 10.0, 800.0),
    ("BMP-GLU", "Glucose", "mg/dL", 70.0, 100.0, 30.0, 500.0),
    ("BMP-BUN", "Blood Urea Nitrogen", "mg/dL", 7.0, 20.0, 2.0, 120.0),
    ("BMP-CRE", "Creatinine", "mg/dL", 0.6, 1.2, 0.2, 15.0),
    ("BMP-NA", "Sodium", "mEq/L", 136.0, 145.0, 110.0, 170.0),
    ("BMP-K", "Potassium", "mEq/L", 3.5, 5.0, 2.0, 8.0),
    ("LFT-ALT", "Alanine Aminotransferase", "U/L", 7.0, 56.0, 1.0, 500.0),
    ("LFT-AST", "Aspartate Aminotransferase", "U/L", 10.0, 40.0, 1.0, 500.0),
    ("LIP-CHOL", "Total Cholesterol", "mg/dL", 0.0, 200.0, 50.0, 400.0),
    ("LIP-HDL", "HDL Cholesterol", "mg/dL", 40.0, 60.0, 10.0, 120.0),
    ("LIP-LDL", "LDL Cholesterol", "mg/dL", 0.0, 100.0, 20.0, 300.0),
    ("A1C", "Hemoglobin A1C", "%", 4.0, 5.6, 3.0, 15.0),
    ("TSH", "Thyroid Stimulating Hormone", "mIU/L", 0.4, 4.0, 0.01, 50.0),
    ("TROP-I", "Troponin I", "ng/mL", 0.0, 0.04, 0.0, 25.0),
    ("CRP", "C-Reactive Protein", "mg/L", 0.0, 3.0, 0.0, 200.0),
    ("D-DIMER", "D-Dimer", "ng/mL", 0.0, 500.0, 0.0, 10000.0),
    ("PT-INR", "INR", "ratio", 0.8, 1.2, 0.5, 10.0)
]

lab_results = []
lab_counter = 0
# Select a random subset of encounters for lab work
lab_encounters = random.sample(enc_ids, min(40000, len(enc_ids)))

for enc_id in lab_encounters:
    # 1-5 lab tests per encounter
    num_labs = random.choices([1, 2, 3, 4, 5], weights=[0.15, 0.25, 0.30, 0.20, 0.10])[0]
    selected_tests = random.sample(lab_tests, min(num_labs, len(lab_tests)))

    for test_code, test_name, unit, ref_low, ref_high, abs_low, abs_high in selected_tests:
        lab_counter += 1
        # 80% normal, 20% abnormal
        if random.random() < 0.80:
            value = round(random.uniform(ref_low, ref_high), 2)
        else:
            value = round(random.uniform(abs_low, abs_high), 2)

        # Determine flag
        if value < ref_low:
            flag = "Low"
        elif value > ref_high:
            flag = "High"
        else:
            flag = "Normal"

        lab_results.append((
            f"LAB-{lab_counter:08d}",                           # lab_result_id
            enc_id,                                              # encounter_id
            enc_patient_map[enc_id],                             # patient_id
            test_code,                                           # test_code
            test_name,                                           # test_name
            value,                                               # result_value
            unit,                                                # unit
            ref_low,                                             # reference_range_low
            ref_high,                                            # reference_range_high
            flag,                                                # abnormal_flag
            (datetime(2021,1,1) + timedelta(days=random.randint(0,1800))).strftime("%Y-%m-%d %H:%M:%S"),  # collected_datetime
            random.choice(["Completed", "Completed", "Completed", "Pending", "Preliminary"])  # status
        ))

lab_schema = StructType([
    StructField("lab_result_id", StringType()),
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("test_code", StringType()),
    StructField("test_name", StringType()),
    StructField("result_value", DoubleType()),
    StructField("unit", StringType()),
    StructField("reference_range_low", DoubleType()),
    StructField("reference_range_high", DoubleType()),
    StructField("abnormal_flag", StringType()),
    StructField("collected_datetime", StringType()),
    StructField("status", StringType())
])

df_lab_results = spark.createDataFrame(lab_results, schema=lab_schema) \
    .withColumn("collected_datetime", F.col("collected_datetime").cast("timestamp"))

print(f"✅ Generated {df_lab_results.count():,} lab result records")
df_lab_results.show(5, truncate=False)

In [0]:
# =============================================================================
# Step 6: Generate ~30,000 medication / prescription records
# =============================================================================
random.seed(321)

medications_ref = [
    ("Lisinopril", "ACE Inhibitor", "10mg", "Oral", "Once daily"),
    ("Metformin", "Biguanide", "500mg", "Oral", "Twice daily"),
    ("Atorvastatin", "Statin", "20mg", "Oral", "Once daily at bedtime"),
    ("Amlodipine", "Calcium Channel Blocker", "5mg", "Oral", "Once daily"),
    ("Metoprolol", "Beta Blocker", "50mg", "Oral", "Twice daily"),
    ("Omeprazole", "Proton Pump Inhibitor", "20mg", "Oral", "Once daily before meals"),
    ("Levothyroxine", "Thyroid Hormone", "50mcg", "Oral", "Once daily on empty stomach"),
    ("Sertraline", "SSRI", "50mg", "Oral", "Once daily"),
    ("Albuterol", "Bronchodilator", "90mcg", "Inhalation", "Every 4-6 hours as needed"),
    ("Amoxicillin", "Penicillin Antibiotic", "500mg", "Oral", "Three times daily"),
    ("Hydrochlorothiazide", "Thiazide Diuretic", "25mg", "Oral", "Once daily"),
    ("Gabapentin", "Anticonvulsant", "300mg", "Oral", "Three times daily"),
    ("Prednisone", "Corticosteroid", "10mg", "Oral", "As directed"),
    ("Warfarin", "Anticoagulant", "5mg", "Oral", "Once daily"),
    ("Insulin Glargine", "Long-acting Insulin", "10 units", "Subcutaneous", "Once daily at bedtime"),
    ("Clopidogrel", "Antiplatelet", "75mg", "Oral", "Once daily"),
    ("Furosemide", "Loop Diuretic", "40mg", "Oral", "Once daily"),
    ("Tramadol", "Opioid Analgesic", "50mg", "Oral", "Every 6 hours as needed"),
    ("Azithromycin", "Macrolide Antibiotic", "250mg", "Oral", "Once daily for 5 days"),
    ("Losartan", "ARB", "50mg", "Oral", "Once daily")
]

med_statuses = ["Active", "Active", "Active", "Discontinued", "Completed", "On Hold"]

medications = []
med_counter = 0
med_encounters = random.sample(enc_ids, min(30000, len(enc_ids)))

for enc_id in med_encounters:
    num_meds = random.choices([1, 2, 3], weights=[0.50, 0.35, 0.15])[0]
    selected_meds = random.sample(medications_ref, min(num_meds, len(medications_ref)))

    for med_name, med_class, dosage, route, frequency in selected_meds:
        med_counter += 1
        start = datetime(2021, 1, 1) + timedelta(days=random.randint(0, 1800))
        duration = random.randint(7, 365)
        end = start + timedelta(days=duration)

        medications.append((
            f"MED-{med_counter:08d}",                           # medication_id
            enc_id,                                              # encounter_id
            enc_patient_map[enc_id],                             # patient_id
            med_name,                                            # medication_name
            med_class,                                           # medication_class
            dosage,                                              # dosage
            route,                                               # route
            frequency,                                           # frequency
            start.strftime("%Y-%m-%d"),                          # start_date
            end.strftime("%Y-%m-%d"),                            # end_date
            f"DR-{random.randint(1000,9999)}",                   # prescribing_physician_id
            random.choice(med_statuses),                         # status
            random.choice([None, "Take with food", "Monitor blood pressure", "Check labs in 2 weeks", 
                          "Avoid grapefruit", "May cause drowsiness"])  # notes
        ))

medication_schema = StructType([
    StructField("medication_id", StringType()),
    StructField("encounter_id", StringType()),
    StructField("patient_id", StringType()),
    StructField("medication_name", StringType()),
    StructField("medication_class", StringType()),
    StructField("dosage", StringType()),
    StructField("route", StringType()),
    StructField("frequency", StringType()),
    StructField("start_date", StringType()),
    StructField("end_date", StringType()),
    StructField("prescribing_physician_id", StringType()),
    StructField("status", StringType()),
    StructField("notes", StringType())
])

df_medications = spark.createDataFrame(medications, schema=medication_schema) \
    .withColumn("start_date", F.col("start_date").cast("date")) \
    .withColumn("end_date", F.col("end_date").cast("date"))

print(f"✅ Generated {df_medications.count():,} medication records")
df_medications.show(5, truncate=False)

In [0]:
# =============================================================================
# Step 7: Write all DataFrames to Unity Catalog as managed tables
# =============================================================================

# Ensure we're in the right catalog/schema
spark.sql("USE CATALOG hipaa_demo")
spark.sql("USE SCHEMA clinical_data")

tables_config = [
    ("patients", df_patients, "Patient demographics including PHI - HIPAA protected"),
    ("encounters", df_encounters, "Hospital encounter/visit records"),
    ("diagnoses", df_diagnoses, "Patient diagnosis records with ICD-10 codes"),
    ("lab_results", df_lab_results, "Laboratory test results"),
    ("medications", df_medications, "Medication and prescription records")
]

for table_name, df, comment in tables_config:
    print(f"Writing {table_name}...")
    df.write \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)
    
    # Add table comment
    spark.sql(f"COMMENT ON TABLE {table_name} IS '{comment}'")
    
    count = spark.sql(f"SELECT COUNT(*) FROM {table_name}").collect()[0][0]
    print(f"  ✅ {table_name}: {count:,} rows written")

print("\n" + "="*60)
print("🎉 ALL TABLES CREATED SUCCESSFULLY!")
print("="*60)

In [0]:
# =============================================================================
# Verify the 'account_managers' account-level group exists before transferring ownership
# =============================================================================
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
group_name = "account_managers"

matching = [g for g in w.groups.list(filter=f'displayName eq "{group_name}"')]

if matching:
    print(f"✅ Account-level group '{group_name}' found. You're good to proceed!")
else:
    raise SystemExit(
        f"❌ Account-level group '{group_name}' NOT found.\n"
        f"Please create it in the Account Console before running the next cell.\n"
        f"See the instructions in the cell above."
    )

In [0]:
%sql
-- =============================================================================
-- Transfer ownership of all tables to the account_managers group
-- This ensures row filters and column masks are evaluated against the
-- querying user's context rather than being bypassed by the table owner.
-- =============================================================================

ALTER TABLE hipaa_demo.clinical_data.patients SET OWNER TO `account_managers`;
ALTER TABLE hipaa_demo.clinical_data.encounters SET OWNER TO `account_managers`;
ALTER TABLE hipaa_demo.clinical_data.diagnoses SET OWNER TO `account_managers`;
ALTER TABLE hipaa_demo.clinical_data.lab_results SET OWNER TO `account_managers`;
ALTER TABLE hipaa_demo.clinical_data.medications SET OWNER TO `account_managers`;

SELECT table_name, table_owner
FROM hipaa_demo.information_schema.tables
WHERE table_schema = 'clinical_data'
ORDER BY table_name;

In [0]:
# =============================================================================
# Grant the current user ALL PRIVILEGES on the catalog, schema, and all tables
# =============================================================================

current_user = spark.sql("SELECT current_user()").collect()[0][0]
print(f"Granting full permissions to: {current_user}")

# Catalog-level
spark.sql(f"GRANT ALL PRIVILEGES ON CATALOG hipaa_demo TO `{current_user}`")
print("✅ Granted ALL PRIVILEGES on CATALOG hipaa_demo")

# Schema-level
spark.sql(f"GRANT ALL PRIVILEGES ON SCHEMA hipaa_demo.clinical_data TO `{current_user}`")
print("✅ Granted ALL PRIVILEGES on SCHEMA hipaa_demo.clinical_data")

# All tables in the schema
tables = [row.table_name for row in spark.sql(
    "SELECT table_name FROM hipaa_demo.information_schema.tables WHERE table_schema = 'clinical_data'"
).collect()]

for table in tables:
    spark.sql(f"GRANT ALL PRIVILEGES ON TABLE hipaa_demo.clinical_data.{table} TO `{current_user}`")
    print(f"✅ Granted ALL PRIVILEGES on TABLE hipaa_demo.clinical_data.{table}")

print(f"\n🎉 Full permissions granted to {current_user} on catalog, schema, and all tables!")

In [0]:
%sql
-- =============================================================================
-- Step 8: Verify all tables exist and show summary statistics
-- =============================================================================

SELECT 
  table_name,
  table_type,
  comment
FROM hipaa_demo.information_schema.tables 
WHERE table_schema = 'clinical_data'
ORDER BY table_name

## Setup Complete!

The following tables are now available in `hipaa_demo.clinical_data`:

| Table | Key Columns | PHI Fields |
|-------|------------|------------|
| `patients` | patient_id, region, primary_department | first_name, last_name, ssn, date_of_birth, address, phone_number, email |
| `encounters` | encounter_id, patient_id, department, region | (linked to patient PHI) |
| `diagnoses` | diagnosis_id, encounter_id, patient_id | (linked to patient PHI) |
| `lab_results` | lab_result_id, encounter_id, patient_id | (linked to patient PHI) |
| `medications` | medication_id, encounter_id, patient_id | (linked to patient PHI) |

### Next: Proceed to the main demo notebook
➡️ **`01 - HIPAA Compliant Patient Analytics Demo`** to implement Unity Catalog security controls, dynamic data masking, row-level security, and audit logging.